# 04 C Gamma and Full Validation

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`EXP_006_SVC_Normal_C_Gamma_SafePilot.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_006_SVC_Normal_C_Gamma_SafePilot.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)
- [`EXP_007_SVC_Normal_Full_Validation.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_007_SVC_Normal_Full_Validation.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/SVC/Normal/scripts/EXP_006_SVC_Normal_C_Gamma_SafePilot.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '28745a6374e14c4bac46ec015f377223cb0bd1f2'}, '05_model_optimize/SVC/Normal/scripts/EXP_007_SVC_Normal_Full_Validation.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '20a3acc70c511867468e2179cfcdc325184c85b2'}}

## EXP_006_SVC_Normal_C_Gamma_SafePilot

원본: [05_model_optimize/SVC/Normal/scripts/EXP_006_SVC_Normal_C_Gamma_SafePilot.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_006_SVC_Normal_C_Gamma_SafePilot.py)  
Git blob: `28745a6374e14c4bac46ec015f377223cb0bd1f2`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)

EXP-006 SVC Normal: RBF C/gamma grid on frozen CoM->Deskew->HOG features.

Run from repository root:
  python 05_model_optimize/SVC/Normal/scripts/EXP_006_SVC_Normal_C_Gamma_SafePilot.py

Safe pilot: 8,000 official train / 3,200 official validation, all 2,560
Hand-made development images. The official Test data are never loaded.

CLI 인자 (정적 추출): `--train-size`, `--val-size`, `--batch-size`

In [ ]:
%%archive_source 05_model_optimize/SVC/Normal/scripts/EXP_006_SVC_Normal_C_Gamma_SafePilot.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-006 SVC Normal: RBF C/gamma grid on frozen CoM->Deskew->HOG features.

Run from repository root:
  python 05_model_optimize/SVC/Normal/scripts/EXP_006_SVC_Normal_C_Gamma_SafePilot.py

Safe pilot: 8,000 official train / 3,200 official validation, all 2,560
Hand-made development images. The official Test data are never loaded.
"""
from __future__ import annotations

import argparse
import csv
import json
import sys
import time
from pathlib import Path

import numpy as np
from sklearn.base import clone

ROOT = Path(__file__).resolve().parents[4]
sys.path[:0] = [
    str(ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'scripts'),
    str(ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'),
    str(ROOT / '03_experiments' / 'model_optimization'),
]
from EXP_001_SVC_Normal_Baseline import WRITERS, baseline, load_writer, report
from EXP_005_SVC_Normal_HOG_SafePilot import predict_batches, select_stratified
from EXP004_KNN_Normal_CoM import BatchedCoM
from EXP011_KNN_Normal_Deskew import deskew, test_deskew
from EXP012_KNN_Normal_HOG import hog_features, test_hog
from common import fixed_split

RESULTS = ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'results'


def say(*args):
    print(*args, flush=True)


def save_json(path, data):
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str), encoding='utf-8')


def run(args):
    svc, scaling, source = baseline()
    if not scaling or svc.kernel != 'rbf':
        raise RuntimeError('Expected the original project SVC with RBF kernel and pixel scaling')
    say('SVC baseline:', source)
    say('Baseline C/gamma:', svc.C, svc.gamma)
    say('EXP-006 SAFE PILOT: CoM -> Deskew -> HOG, C/gamma search')
    say('Official Test NOT loaded. Hand-made is repeated-use DEVELOPMENT data.')
    test_deskew()
    test_hog()

    t = time.perf_counter()
    xtr, ytr, xval, yval, sha = fixed_split()
    say(f'Official split loaded in {time.perf_counter() - t:.2f}s: {len(ytr)} / {len(yval)}')
    xtr, ytr = select_stratified(xtr, ytr, args.train_size)
    xval, yval = select_stratified(xval, yval, args.val_size)
    writers = {w: load_writer(w) for w in WRITERS}
    say(f'Pilot train={len(ytr)}, validation={len(yval)}, handmade={sum(len(writers[w][1]) for w in WRITERS)}')

    t = time.perf_counter()
    com = BatchedCoM()
    centered_train = com.transform(xtr)
    centered_val = com.transform(xval)
    centered_writers = {w: com.transform(writers[w][0]) for w in WRITERS}
    say(f'CoM preprocessed in {time.perf_counter() - t:.2f}s')

    t = time.perf_counter()
    pixel_train, _ = deskew(centered_train)
    pixel_val, _ = deskew(centered_val)
    pixel_writers = {w: deskew(centered_writers[w])[0] for w in WRITERS}
    say(f'Deskew preprocessed in {time.perf_counter() - t:.2f}s')

    t = time.perf_counter()
    hog_train = np.ascontiguousarray(hog_features(pixel_train), dtype=np.float32)
    hog_val = np.ascontiguousarray(hog_features(pixel_val), dtype=np.float32)
    hog_writers = {w: np.ascontiguousarray(hog_features(pixel_writers[w]), dtype=np.float32) for w in WRITERS}
    if hog_train.shape != (len(ytr), 324) or not np.isfinite(hog_train).all():
        raise RuntimeError(f'Unexpected HOG features: {hog_train.shape}')
    say(f'HOG features prepared in {time.perf_counter() - t:.2f}s; dims={hog_train.shape[1]}')

    output_dir = RESULTS / f'EXP_006_SVC_Normal_C_Gamma_SAFE_PILOT_train{len(ytr)}'
    output_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    records = {}
    c_values = [0.1, 1.0, 10.0]
    gamma_values = ['scale', 0.01, 0.1]

    for c in c_values:
        for gamma in gamma_values:
            name = f'C_{c:g}_gamma_{gamma}'
            model = clone(svc).set_params(C=c, gamma=gamma, cache_size=300)
            say(f'\n[{name}] fit START rows={len(ytr)} dims=324')
            t = time.perf_counter()
            model.fit(hog_train, ytr)
            fit_sec = round(time.perf_counter() - t, 2)
            say(f'[{name}] fit DONE {fit_sec}s')
            official_pred, official_sec = predict_batches(model, hog_val, f'{name} official', args.batch_size)
            official = report(yval, official_pred)
            say(f'[{name}] official={official["accuracy"]:.6f} F1={official["macro_f1"]:.6f} predict={official_sec}s')
            ys, ps, writer_rows = [], [], []
            for w in WRITERS:
                pred, sec = predict_batches(model, hog_writers[w], f'{name} writer {w}', args.batch_size)
                target = writers[w][1]
                wr = report(target, pred)
                writer_rows.append({'writer': w, 'accuracy': wr['accuracy'], 'macro_f1': wr['macro_f1'], 'n': len(target), 'predict_seconds': sec})
                ys.append(target)
                ps.append(pred)
                say(f'[{name}] writer {w} accuracy={wr["accuracy"]:.6f}')
            handmade = report(np.concatenate(ys), np.concatenate(ps))
            say(f'[{name}] Hand-made={handmade["accuracy"]:.6f} F1={handmade["macro_f1"]:.6f}')
            rec = {'C': c, 'gamma': gamma, 'fit_seconds': fit_sec,
                   'official_predict_seconds': official_sec, 'official_validation': official,
                   'handmade_development': handmade, 'writers': writer_rows}
            records[name] = rec
            rows.append({'candidate': name, 'C': c, 'gamma': gamma, 'train_size': len(ytr),
                         'fit_seconds': fit_sec, 'official_accuracy': official['accuracy'],
                         'official_macro_f1': official['macro_f1'],
                         'handmade_accuracy': handmade['accuracy'],
                         'handmade_macro_f1': handmade['macro_f1']})
            save_json(output_dir / 'metrics_partial.json', records)
            with (output_dir / 'comparison_partial.csv').open('w', encoding='utf-8-sig', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=list(rows[0]))
                writer.writeheader()
                writer.writerows(rows)
            del model

    control = records['C_1_gamma_scale']
    if args.train_size == 8000 and args.val_size == 3200:
        expected = (0.9753125, 0.89375)
        found = (control['official_validation']['accuracy'], control['handmade_development']['accuracy'])
        if not np.allclose(found, expected, atol=1e-9):
            raise RuntimeError(f'EXP-005 HOG control mismatch: found={found}; expected={expected}')
        say('EXP-005 HOG CONTROL EXACTLY REPRODUCED')

    # Development choice prioritizes Hand-made macro F1; use official accuracy to break ties.
    best = max(rows, key=lambda r: (r['handmade_macro_f1'], r['official_accuracy']))
    result = {'experiment': 'EXP-006 SVC Normal RBF C/gamma safe pilot',
              'train_size': len(ytr), 'validation_size': len(yval), 'official_test_used': False,
              'model_source': source, 'baseline_params': svc.get_params(),
              'official_dataset_sha256': sha, 'preprocessing': 'CoM -> Deskew -> HOG(324)',
              'feature_scaling': 'HOG pre-normalized; no pixel/255 scaling',
              'results': records, 'selected_development_candidate': best['candidate'],
              'note': 'Hand-made data used repeatedly for development, not an independent test.'}
    save_json(output_dir / 'metrics.json', result)
    with (output_dir / 'comparison.csv').open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    say('\nEXP-006 SAFE PILOT SUMMARY')
    for row in rows:
        say(f'  {row["candidate"]:<24} official={row["official_accuracy"]:.6f} '
            f'handmade={row["handmade_accuracy"]:.6f} F1={row["handmade_macro_f1"]:.6f}')
    say('SELECTED DEVELOPMENT CANDIDATE:', best['candidate'])
    say('Saved:', output_dir)
    say('Official Test was NOT loaded.')


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--train-size', type=int, default=8000, choices=(2000, 4000, 8000, 16000))
    parser.add_argument('--val-size', type=int, default=3200, choices=(800, 1600, 3200, 6400))
    parser.add_argument('--batch-size', type=int, default=256, choices=(64, 128, 256, 512))
    run(parser.parse_args())


## EXP_007_SVC_Normal_Full_Validation

원본: [05_model_optimize/SVC/Normal/scripts/EXP_007_SVC_Normal_Full_Validation.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_007_SVC_Normal_Full_Validation.py)  
Git blob: `20a3acc70c511867468e2179cfcdc325184c85b2`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)

SVC Normal EXP-007: full official Training, HOG or optional Pixel:HOG.

From the repository root:
  python 05_model_optimize/SVC/Normal/scripts/EXP_007_SVC_Normal_Full_Validation.py
  python 05_model_optimize/SVC/Normal/scripts/EXP_007_SVC_Normal_Full_Validation.py --feature hybrid

Only official Training/Validation, never Test. This script does not alter prior results.

CLI 인자 (정적 추출): `--feature`, `--batch-size`, `--cache-size`, `--overwrite`

In [ ]:
%%archive_source 05_model_optimize/SVC/Normal/scripts/EXP_007_SVC_Normal_Full_Validation.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""SVC Normal EXP-007: full official Training, HOG or optional Pixel:HOG.

From the repository root:
  python 05_model_optimize/SVC/Normal/scripts/EXP_007_SVC_Normal_Full_Validation.py
  python 05_model_optimize/SVC/Normal/scripts/EXP_007_SVC_Normal_Full_Validation.py --feature hybrid

Only official Training/Validation, never Test. This script does not alter prior results.
"""
from __future__ import annotations
import argparse
import csv
import json
import os
import sys
import time
import gc
from pathlib import Path

import numpy as np
from sklearn.base import clone
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

ROOT = Path(__file__).resolve().parents[4]
sys.path[:0] = [
    str(ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'scripts'),
    str(ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'),
    str(ROOT / '03_experiments' / 'model_optimization'),
]
from EXP_001_SVC_Normal_Baseline import WRITERS, baseline, load_writer, report
from EXP_005_SVC_Normal_HOG_SafePilot import predict_batches
from EXP004_KNN_Normal_CoM import BatchedCoM
from EXP011_KNN_Normal_Deskew import deskew, test_deskew
from EXP012_KNN_Normal_HOG import hog_features, hybrid_features, test_hog
from common import fixed_split, LABELS

OUT = ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'results' / 'EXP_007_SVC_Normal_FULL_76800'


def say(*items):
    print(*items, flush=True)


def save_json(path, content):
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(content, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    os.replace(temp, path)


def save_csv(path, records, columns):
    with path.open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows(records)


def features(x, feature, title):
    start = time.perf_counter()
    com = BatchedCoM().transform(x)
    say(f'[{title}] CoM {time.perf_counter() - start:.2f}s')
    start = time.perf_counter()
    pixel, _ = deskew(com)
    del com
    say(f'[{title}] Deskew {time.perf_counter() - start:.2f}s')
    start = time.perf_counter()
    hog = hog_features(pixel)
    say(f'[{title}] HOG {time.perf_counter() - start:.2f}s')
    if feature == 'hybrid':
        start = time.perf_counter()
        feat = hybrid_features(pixel, hog, pixel_weight=0.20)
        say(f'[{title}] Hybrid 20:80 {time.perf_counter() - start:.2f}s')
    else:
        feat = np.ascontiguousarray(hog, dtype=np.float32)
    del pixel, hog
    gc.collect()
    if feat.dtype != np.float32 or not np.isfinite(feat).all():
        raise RuntimeError(f'{title}: invalid features')
    return feat


def run(args):
    svc, scaling, source = baseline()
    if not scaling or svc.kernel != 'rbf':
        raise RuntimeError('Unexpected team baseline: expected pixel-scaled RBF SVC')
    test_deskew()
    test_hog()
    say('EXP-007: FULL official training 76800 + validation 19200')
    say('Feature:', args.feature, '| SVC: C=1, gamma=scale, kernel=rbf')
    say('Official Test is NOT loaded. Hand-made is DEVELOPMENT only.')
    say('Original SVC source:', source)
    OUT.mkdir(parents=True, exist_ok=True)
    path = OUT / f'{args.feature}_metrics.json'
    if path.exists() and not args.overwrite:
        raise FileExistsError(f'Results already exist: {path}. Use --overwrite to rerun intentionally.')
    t = time.perf_counter()
    xtr, ytr, xval, yval, sha = fixed_split()
    if (len(ytr), len(yval)) != (76800, 19200):
        raise RuntimeError('Official split mismatch; refusing partial experiment')
    writers = {w: load_writer(w) for w in WRITERS}
    say(f'Data loaded in {time.perf_counter()-t:.2f}s; {len(ytr)} / {len(yval)} + handmade 2560')
    status = {'experiment': 'EXP-007 SVC Normal full training', 'feature': args.feature,
              'official_training': len(ytr), 'official_validation_size': len(yval),
              'official_test_used': False, 'dataset_sha256': sha,
              'baseline_source': source, 'parameters': {'kernel':'rbf','C':1.0,'gamma':'scale',
                  'cache_size_mb': args.cache_size}, 'status':'preprocessing',
              'note': 'Hand-made evaluation is repeated-use development data.'}
    save_json(OUT / f'{args.feature}_status.json', status)
    train = features(xtr, args.feature, 'Train 76800')
    del xtr
    gc.collect()
    val = features(xval, args.feature, 'Validation 19200')
    del xval
    hw = {}
    for w in WRITERS:
        x, y, _ = writers[w]
        hw[w] = (features(x, args.feature, f'Writer {w}'), y)
    del writers
    gc.collect()
    say(f'Prepared train={train.shape}, val={val.shape}, dtype={train.dtype}')
    status['status'] = 'fitting'
    status['feature_dimensions'] = int(train.shape[1])
    save_json(OUT / f'{args.feature}_status.json', status)
    model = clone(svc).set_params(C=1.0, gamma='scale', cache_size=args.cache_size)
    say('SVC fit START (full dataset; RBF optimization may take substantial CPU/RAM)')
    t = time.perf_counter()
    model.fit(train, ytr)
    fit_seconds = round(time.perf_counter()-t, 2)
    say(f'SVC fit DONE: {fit_seconds:.2f}s; support_vectors={len(model.support_)}')
    status.update(status='trained', fit_seconds=fit_seconds,
                  support_vectors=int(len(model.support_)))
    save_json(OUT / f'{args.feature}_status.json', status)
    del train
    gc.collect()
    pred, val_seconds = predict_batches(model, val, 'Official Validation', args.batch_size)
    official = report(yval, pred)
    say(f'Official Validation: accuracy={official["accuracy"]:.6f} F1={official["macro_f1"]:.6f}')
    status.update(status='official_evaluated', official_validation=official,
                  validation_predict_seconds=val_seconds)
    save_json(OUT / f'{args.feature}_status.json', status)
    del val
    gc.collect()
    yt, yp, writer_rows, pred_rows = [], [], [], []
    for w in WRITERS:
        x, y = hw[w]
        hp, htime = predict_batches(model, x, f'Writer {w}', args.batch_size)
        stats = report(y, hp)
        writer_rows.append({'writer':w, **stats, 'predict_seconds': htime})
        say(f'Writer {w}: {stats["correct"]}/640 accuracy={stats["accuracy"]:.6f}')
        pred_rows.extend({'writer':w,'index':i,'true':LABELS[int(a)],
                          'predicted': LABELS[int(b)], 'correct':int(a==b)}
                         for i,(a,b) in enumerate(zip(y,hp)))
        yt.append(y)
        yp.append(hp)
    yt = np.concatenate(yt)
    yp = np.concatenate(yp)
    handmade = report(yt,yp)
    cm = confusion_matrix(yt,yp,labels=np.arange(16))
    pr, re, f1, support = precision_recall_fscore_support(
        yt,yp,labels=np.arange(16),zero_division=0)
    save_csv(OUT / f'{args.feature}_writers.csv',writer_rows,
             ['writer','n','correct','accuracy','macro_f1','predict_seconds'])
    save_csv(OUT / f'{args.feature}_handmade_predictions.csv',pred_rows,
             ['writer','index','true','predicted','correct'])
    save_csv(OUT / f'{args.feature}_handmade_confusion_matrix.csv',
             [{'true': LABELS[i], **{LABELS[j]:int(cm[i,j]) for j in range(16)}} for i in range(16)],
             ['true',*LABELS])
    save_csv(OUT / f'{args.feature}_per_class.csv',
             [{'class':LABELS[i],'precision':float(pr[i]),'recall':float(re[i]),
               'f1':float(f1[i]),'support':int(support[i])} for i in range(16)],
             ['class','precision','recall','f1','support'])
    status.update(status='complete', handmade_development=handmade, writers=writer_rows)
    save_json(path,status)
    save_json(OUT / f'{args.feature}_status.json',status)
    say(f'FULL EXP-007 {args.feature}: official={official["accuracy"]:.6f} '
        f'handmade={handmade["accuracy"]:.6f} macro_f1={handmade["macro_f1"]:.6f} '
        f'({handmade["correct"]}/2560)')
    say('Results saved:',OUT)
    say('Official Test was NOT loaded.')


if __name__ == '__main__':
    parser=argparse.ArgumentParser()
    parser.add_argument('--feature',choices=['hog','hybrid'],default='hog')
    parser.add_argument('--batch-size',type=int,default=256,choices=[64,128,256,512])
    parser.add_argument('--cache-size',type=int,default=200,help='LibSVM cache size in MB')
    parser.add_argument('--overwrite',action='store_true')
    run(parser.parse_args())


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")